# 🔍 UPI Fraud Detection — Exploratory Data Analysis

> **Project Goal:** Understand the structure and patterns (or lack thereof) in UPI transaction data to determine the feasibility of building a fraud detection model.

---

## 📌 Dataset Overview
This dataset contains UPI (Unified Payments Interface) transactions labeled as fraudulent or genuine. Each transaction includes attributes like amount, device type, network, merchant category, time-of-day, and geographic information.

## 🎯 Key Finding (Spoiler)
> ⚠️ After thorough EDA, **no significant statistical difference** was found between fraud and non-fraud transactions across most features. This is documented in detail below and highlights the challenge of detecting subtle fraud in UPI systems.

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Basic Data Inspection](#2)
3. [Missing Values & Duplicates](#3)
4. [Target Variable Analysis](#4)
5. [Transaction Amount Analysis](#5)
6. [Categorical Feature Analysis](#6)
7. [Time-Based Analysis](#7)
8. [Geographic Analysis](#8)
9. [Correlation & Feature Relationships](#9)
10. [Key Findings & Conclusion](#10)

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('✅ Libraries loaded successfully')

In [ ]:
df = pd.read_csv('../data/upi_fraud.csv')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 2. Basic Data Inspection <a id='2'></a>

Let's understand the schema — column names, data types, and a statistical summary.

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Column breakdown by dtype
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Numerical columns ({len(num_cols)}): {num_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

---
## 3. Missing Values & Duplicates <a id='3'></a>

Data quality check — nulls and duplicate rows must be identified before any analysis.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found!')
else:
    print(missing_df)

print(f'\n🔁 Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Visualise missing values (if any)
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_df['Missing %'].plot(kind='barh', ax=ax, color='coral')
    ax.set_title('Missing Value % by Column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing value chart needed — dataset is clean.')

---
## 4. Target Variable Analysis <a id='4'></a>

Understanding the class distribution of `fraud_flag` (1 = Fraud, 0 = Genuine). Class imbalance is a critical concern in fraud detection.

In [ ]:
fraud_counts = df['fraud_flag'].value_counts()
fraud_pct = df['fraud_flag'].value_counts(normalize=True) * 100

print('--- Class Distribution ---')
for label, count in fraud_counts.items():
    tag = 'Fraud' if label == 1 else 'Genuine'
    print(f'  {tag} ({label}): {count:,} ({fraud_pct[label]:.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
labels = ['Genuine (0)', 'Fraud (1)']
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(labels, fraud_counts.values, color=colors, width=0.5, edgecolor='black')
axes[0].set_title('Fraud vs Genuine — Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 5, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    fraud_counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Balance (Pie Chart)', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: fraud_flag', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **📝 Observation:** The chart above shows the class split. If highly imbalanced, standard accuracy is a misleading metric — precision, recall, and F1 would be better choices for modelling.

---
## 5. Transaction Amount Analysis <a id='5'></a>

Amount is typically one of the strongest fraud signals. Here we compare distributions between fraud and genuine transactions.

In [ ]:
print('=== Overall Amount Statistics ===')
print(df['amount (INR)'].describe().round(2))
print('\n=== Amount Statistics by Fraud Flag ===')
print(df.groupby('fraud_flag')['amount (INR)'].describe().round(2))

In [ ]:
# Mann-Whitney U test to check if distributions differ significantly
fraud_amt = df[df['fraud_flag'] == 1]['amount (INR)']
genuine_amt = df[df['fraud_flag'] == 0]['amount (INR)']
stat, p_val = stats.mannwhitneyu(fraud_amt, genuine_amt, alternative='two-sided')
print(f'Mann-Whitney U Test — p-value: {p_val:.4f}')
if p_val < 0.05:
    print('✅ Statistically significant difference in amounts between classes.')
else:
    print('⚠️  No statistically significant difference in amounts between classes.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Overall distribution
sns.histplot(df['amount (INR)'], bins=60, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Overall Transaction Amount Distribution')
axes[0, 0].set_xlabel('Amount (INR)')

# 2. KDE by fraud flag
for flag, color, label in [(0, '#2ecc71', 'Genuine'), (1, '#e74c3c', 'Fraud')]:
    sns.kdeplot(df[df['fraud_flag'] == flag]['amount (INR)'],
                ax=axes[0, 1], label=label, color=color, fill=True, alpha=0.4)
axes[0, 1].set_title('Amount Distribution: Fraud vs Genuine (KDE)')
axes[0, 1].legend()

# 3. Boxplot
sns.boxplot(x='fraud_flag', y='amount (INR)', data=df, ax=axes[1, 0],
            palette={0: '#2ecc71', 1: '#e74c3c'})
axes[1, 0].set_title('Boxplot: Amount by Fraud Flag')
axes[1, 0].set_xticklabels(['Genuine', 'Fraud'])

# 4. Log-scale boxplot (better for skewed data)
sns.boxplot(x='fraud_flag', y='amount (INR)', data=df, ax=axes[1, 1],
            palette={0: '#2ecc71', 1: '#e74c3c'})
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Boxplot: Amount by Fraud Flag (Log Scale)')
axes[1, 1].set_xticklabels(['Genuine', 'Fraud'])

plt.suptitle('Transaction Amount Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **📝 Observation:** If the KDE curves heavily overlap, it means amount alone is not a strong differentiator between fraud and genuine transactions — consistent with the dataset's pattern problem.

---
## 6. Categorical Feature Analysis <a id='6'></a>

We examine how fraud is distributed across transaction type, device type, network type, and merchant category.

In [ ]:
cat_features = ['transaction type', 'device_type', 'network_type', 'merchant_category']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    if col not in df.columns:
        axes[i].set_visible(False)
        continue
    fraud_rate = df.groupby(col)['fraud_flag'].mean() * 100
    fraud_rate = fraud_rate.sort_values(ascending=False)
    bars = axes[i].bar(fraud_rate.index, fraud_rate.values,
                       color=plt.cm.RdYlGn_r(fraud_rate.values / fraud_rate.max()),
                       edgecolor='black')
    axes[i].set_title(f'Fraud Rate by {col} (%)', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Fraud Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].axhline(y=df['fraud_flag'].mean() * 100, color='blue',
                    linestyle='--', label='Overall avg', linewidth=1.5)
    axes[i].legend(fontsize=9)
    for bar, val in zip(bars, fraud_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Fraud Rate Across Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar — volume breakdown
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    if col not in df.columns:
        axes[i].set_visible(False)
        continue
    ct = pd.crosstab(df[col], df['fraud_flag'], normalize='index') * 100
    ct.columns = ['Genuine', 'Fraud']
    ct[['Genuine', 'Fraud']].plot(
        kind='bar', stacked=True, ax=axes[i],
        color=['#2ecc71', '#e74c3c'], edgecolor='black'
    )
    axes[i].set_title(f'{col} — Fraud vs Genuine %', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(loc='upper right')

plt.suptitle('Fraud vs Genuine Split (Stacked) by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Chi-square tests for categorical features
print('=== Chi-Square Test Results (Feature vs Fraud Flag) ===')
print(f'{"Feature":<25} {"Chi2":>10} {"p-value":>12} {"Significant?":>15}')
print('-' * 65)
for col in cat_features:
    if col not in df.columns:
        continue
    ct = pd.crosstab(df[col], df['fraud_flag'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = '✅ Yes' if p < 0.05 else '❌ No'
    print(f'{col:<25} {chi2:>10.2f} {p:>12.4f} {sig:>15}')

> **📝 Observation:** If p-values are all > 0.05, categorical features are not statistically associated with fraud — this is the core problem with this dataset. Every category has nearly the same fraud rate, making pattern learning extremely difficult.

---
## 7. Time-Based Analysis <a id='7'></a>

Fraudulent transactions often cluster at unusual hours (late night / early morning). Let's investigate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud count by hour
hourly = df.groupby('hour_of_day')['fraud_flag'].agg(['sum', 'count'])
hourly['fraud_rate'] = hourly['sum'] / hourly['count'] * 100

axes[0].bar(hourly.index, hourly['sum'], color='#e74c3c', edgecolor='black')
axes[0].set_title('Fraud Count by Hour of Day', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour (0 = Midnight)')
axes[0].set_ylabel('Fraud Count')
axes[0].set_xticks(range(0, 24))

# Fraud rate by hour
axes[1].plot(hourly.index, hourly['fraud_rate'], marker='o',
             color='#e67e22', linewidth=2, markersize=5)
axes[1].axhline(y=hourly['fraud_rate'].mean(), linestyle='--',
                color='gray', label='Avg rate')
axes[1].set_title('Fraud Rate (%) by Hour of Day', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hour (0 = Midnight)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xticks(range(0, 24))
axes[1].legend()

plt.suptitle('Temporal Analysis — Hour of Day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Weekend analysis
if 'is_weekend' in df.columns:
    weekend_fraud = pd.crosstab(df['is_weekend'], df['fraud_flag'], normalize='index') * 100
    weekend_fraud.index = ['Weekday', 'Weekend']
    weekend_fraud.columns = ['Genuine %', 'Fraud %']
    print('=== Weekend vs Weekday Fraud Rate ===')
    print(weekend_fraud.round(2))

    fig, ax = plt.subplots(figsize=(7, 4))
    weekend_fraud[['Genuine %', 'Fraud %']].plot(
        kind='bar', ax=ax,
        color=['#2ecc71', '#e74c3c'],
        edgecolor='black'
    )
    ax.set_title('Fraud Rate: Weekend vs Weekday', fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

---
## 8. Geographic Analysis <a id='8'></a>

Which states have the highest absolute fraud count and the highest fraud rate?

In [ ]:
if 'sender_state' in df.columns:
    state_stats = df.groupby('sender_state')['fraud_flag'].agg(['sum', 'count'])
    state_stats.columns = ['fraud_count', 'total']
    state_stats['fraud_rate'] = state_stats['fraud_count'] / state_stats['total'] * 100
    state_stats = state_stats.sort_values('fraud_count', ascending=False)

    top10_count = state_stats.head(10)
    top10_rate = state_stats.sort_values('fraud_rate', ascending=False).head(10)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Top by count
    axes[0].barh(top10_count.index[::-1], top10_count['fraud_count'][::-1],
                 color='#c0392b', edgecolor='black')
    axes[0].set_title('Top 10 States — Fraud Transaction Count', fontweight='bold')
    axes[0].set_xlabel('Fraud Count')

    # Top by rate
    axes[1].barh(top10_rate.index[::-1], top10_rate['fraud_rate'][::-1],
                 color='#e67e22', edgecolor='black')
    axes[1].set_title('Top 10 States — Fraud Rate (%)', fontweight='bold')
    axes[1].set_xlabel('Fraud Rate (%)')
    axes[1].axvline(x=df['fraud_flag'].mean() * 100, color='blue',
                    linestyle='--', label='Avg rate')
    axes[1].legend()

    plt.suptitle('Geographic Distribution of Fraud', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 9. Correlation & Feature Relationships <a id='9'></a>

A correlation heatmap and pairplot to see if any numeric features relate to each other or to the fraud flag.

In [ ]:
num_cols_available = [c for c in ['amount (INR)', 'fraud_flag', 'hour_of_day', 'is_weekend']
                      if c in df.columns]
corr = df[num_cols_available].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            vmin=-1, vmax=1, center=0)
ax.set_title('Correlation Heatmap (Numerical Features)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with fraud_flag specifically
print('=== Pearson Correlation with fraud_flag ===')
corr_with_target = df[num_cols_available].corr()['fraud_flag'].drop('fraud_flag').sort_values()
print(corr_with_target.round(4))

fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with fraud_flag', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plot — amount by fraud and transaction type combined
if 'transaction type' in df.columns:
    plt.figure(figsize=(14, 5))
    sns.violinplot(
        x='transaction type', y='amount (INR)', hue='fraud_flag',
        data=df, split=True,
        palette={0: '#2ecc71', 1: '#e74c3c'}
    )
    plt.title('Amount Distribution by Transaction Type and Fraud Flag (Violin)', fontweight='bold')
    plt.xticks(rotation=30)
    plt.legend(title='fraud_flag', labels=['Genuine', 'Fraud'])
    plt.tight_layout()
    plt.show()

---
## 10. Key Findings & Conclusion <a id='10'></a>

### 🔍 Summary of Findings

| Feature | Finding |
|---|---|
| **Transaction Amount** | Distribution nearly identical for fraud vs genuine — weak signal |
| **Transaction Type** | Fraud rate uniform across types — no distinguishing pattern |
| **Device Type** | No meaningful difference in fraud rate |
| **Network Type** | No meaningful difference in fraud rate |
| **Hour of Day** | Fraud count follows transaction volume — no late-night spike |
| **Weekend** | Negligible difference between weekday and weekend fraud |
| **Merchant Category** | Uniform fraud rates across categories |
| **Geography** | Fraud count correlates with state size, not actual fraud rate |

### ⚠️ Core Problem: No Separability

The most critical finding of this EDA is that **fraudulent and genuine transactions are statistically indistinguishable** across all available features. This suggests:

1. **The dataset may be synthetically generated** with equal-distribution assumptions — a common issue with public fraud datasets.
2. **Feature engineering is needed** — raw features lack signal; derived features (velocity, behavioral sequence, IP mismatch, etc.) would be needed.
3. **A standard supervised ML model would struggle** — without feature separation, the model would likely default to predicting the majority class.

### ✅ What This EDA Proves

> A complete EDA doesn't always end with a model-ready dataset. Sometimes the most valuable output is **knowing what won't work and why**. This analysis rigorously documents that limitation — which is itself a meaningful contribution.

### 🚀 Potential Next Steps
- Explore **graph-based fraud detection** using transaction networks
- Combine with **external behavioral data** (login attempts, device fingerprint)
- Try **anomaly detection** (Isolation Forest, Autoencoders) instead of supervised learning
- Look for a **richer dataset** with more behavioural and session-level features

In [ ]:
print('='*55)
print('   UPI Fraud EDA — Completed Successfully')
print('='*55)
print(f'  Total Transactions Analysed : {len(df):,}')
print(f'  Features Examined           : {len(df.columns)}')
print(f'  Fraud Transactions          : {df["fraud_flag"].sum():,}')
print(f'  Genuine Transactions        : {(df["fraud_flag"]==0).sum():,}')
print(f'  Overall Fraud Rate          : {df["fraud_flag"].mean()*100:.2f}%')
print('='*55)
print('  ⚠️  Conclusion: Low feature separability detected.')
print('     Standard ML models unlikely to generalise.')
print('='*55)